# Random Variables

A **random variable** (RV) is a numerical quantity whose value is determined by a random experiment.
Think of it as a variable in a programming language — except its value is assigned by chance.

**Two families:**
| Type | Values | Example |
|------|--------|---------|
| **Discrete** | Countable (integers) | Number of heads in 10 flips |
| **Continuous** | Real numbers | Exact arrival time of a bus |

**Key properties** of every random variable:
| Property | Meaning |
|----------|---------|
| Support / Range | Values the RV can take on |
| PMF or PDF | Likelihood function |
| Expectation $E[X]$ | Weighted average |
| Variance $\text{Var}(X)$ | Measure of spread |
| Mode | Most likely value |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("Libraries loaded!")

---
## 1. Defining a Random Variable — Coin Flip Example

Let $Y$ = number of heads when flipping 3 fair coins.

$Y$ can take values $\{0, 1, 2, 3\}$. Each outcome maps to a number:

| Outcome | $Y$ |
|---------|-----|
| TTT | 0 |
| HTT, THT, TTH | 1 |
| HHT, HTH, THH | 2 |
| HHH | 3 |

In [ ]:
# Enumerate all outcomes of 3 coin flips
from itertools import product

outcomes = list(product(['H', 'T'], repeat=3))
Y_values = {o: o.count('H') for o in outcomes}

print("Outcome → Y (number of heads)")
print("-" * 30)
for outcome, y in sorted(Y_values.items(), key=lambda x: x[1]):
    print(f"  {''.join(outcome):>3s}  →  {y}")

print(f"\nSample space size: {len(outcomes)}")
print(f"Possible values of Y: {sorted(set(Y_values.values()))}")

In [ ]:
# Visualize: mapping from outcomes to random variable values
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.set_xlim(-1, 11)
ax.set_ylim(-0.5, 4.5)
ax.axis('off')
ax.set_title('Random Variable: Mapping Outcomes → Numbers', fontsize=14, fontweight='bold')

# Group outcomes by Y value
from collections import defaultdict
groups = defaultdict(list)
for o, y in Y_values.items():
    groups[y].append(''.join(o))

colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
x_start = 0.5

for y_val in sorted(groups):
    members = groups[y_val]
    y_pos = 3.5 - y_val * 1.0
    
    # Draw the Y value box on the right
    rect = patches.FancyBboxPatch((8, y_pos - 0.25), 1.2, 0.5,
                                   boxstyle='round,pad=0.1',
                                   facecolor=colors[y_val], alpha=0.8,
                                   edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(8.6, y_pos, f'Y = {y_val}', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')
    
    # Draw outcome boxes on the left
    for i, m in enumerate(members):
        x = x_start + i * 2.0
        rect = patches.FancyBboxPatch((x, y_pos - 0.2), 1.2, 0.4,
                                       boxstyle='round,pad=0.05',
                                       facecolor=colors[y_val], alpha=0.3,
                                       edgecolor=colors[y_val], linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + 0.6, y_pos, m, ha='center', va='center', fontsize=10, fontweight='bold')
        # Arrow from outcome to Y value
        ax.annotate('', xy=(8, y_pos), xytext=(x + 1.2, y_pos),
                    arrowprops=dict(arrowstyle='->', color=colors[y_val], lw=1.5, alpha=0.5))

ax.text(3, 4.2, 'Outcomes (sample space)', ha='center', fontsize=12, style='italic')
ax.text(8.6, 4.2, 'RV values', ha='center', fontsize=12, style='italic')
plt.tight_layout()
plt.show()

---
## 2. Random Variables vs Events

**Events** are outcomes or sets of outcomes. **Random variables** are numerical mappings of experiments.

To get a **probability** from an RV, you must create an **event** using relational operators:

| Expression | Meaning |
|-----------|---------|
| $P(Y = 1)$ | Probability of exactly 1 head |
| $P(Y \leq 1)$ | Probability of 0 or 1 heads |
| $P(Y > 2)$ | Probability of more than 2 heads |
| $P(Y = y)$ | General: probability of exactly $y$ heads |

In [ ]:
# Compute probabilities for each event
from fractions import Fraction

total = len(outcomes)
print("Event probabilities for Y = # heads in 3 fair coin flips\n")

for y in range(4):
    count = sum(1 for o in outcomes if o.count('H') == y)
    prob = Fraction(count, total)
    matching = [' '.join(o) for o in outcomes if o.count('H') == y]
    print(f"P(Y = {y}) = {count}/{total} = {float(prob):.4f}    ← {matching}")

print(f"\nP(Y ≤ 1) = P(Y=0) + P(Y=1) = {Fraction(1,8) + Fraction(3,8)} = {float(Fraction(1,8) + Fraction(3,8)):.4f}")
print(f"P(Y > 2) = P(Y=3)           = {Fraction(1,8)} = {float(Fraction(1,8)):.4f}")
print(f"P(Y ≥ 4) = 0                (impossible with 3 coins)")

In [ ]:
# Visualize the probability distribution of Y
y_vals = [0, 1, 2, 3]
probs = [1/8, 3/8, 3/8, 1/8]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PMF bar chart
ax = axes[0]
bars = ax.bar(y_vals, probs, color=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'],
              edgecolor='black', linewidth=1.5, width=0.6)
ax.set_xlabel('y (number of heads)', fontsize=12)
ax.set_ylabel('P(Y = y)', fontsize=12)
ax.set_title('Probability Distribution of Y', fontsize=13, fontweight='bold')
ax.set_xticks(y_vals)
ax.set_ylim(0, 0.5)
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{p:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Right: Simulation vs theory
ax = axes[1]
np.random.seed(42)
n_sims = 10000
simulations = np.sum(np.random.choice([0, 1], size=(n_sims, 3)), axis=1)
sim_probs = [np.mean(simulations == y) for y in y_vals]

x = np.arange(len(y_vals))
w = 0.35
ax.bar(x - w/2, probs, w, label='Theory', color='#3498db', edgecolor='black', alpha=0.8)
ax.bar(x + w/2, sim_probs, w, label=f'Simulation (n={n_sims:,})', color='#e74c3c', edgecolor='black', alpha=0.8)
ax.set_xlabel('y (number of heads)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Theory vs Simulation', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(y_vals)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## 3. Discrete vs Continuous Random Variables

| | Discrete | Continuous |
|---|---|---|
| **Values** | Countable (integers) | Uncountable (reals) |
| **Likelihood function** | PMF: $P(X = x)$ | PDF: $f(x)$ |
| **Probability of exact value** | Can be $> 0$ | Always 0 |
| **Probability of range** | $\sum_{x=a}^{b} P(X=x)$ | $\int_a^b f(x)\,dx$ |
| **Examples** | Dice, coin flips, counts | Height, weight, time |

In [ ]:
# Side-by-side: discrete vs continuous
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Discrete: sum of two dice
ax = axes[0]
x_disc = np.arange(2, 13)
pmf_dice = np.array([1, 2, 3, 4, 5, 6, 5, 4, 3, 2, 1]) / 36
ax.bar(x_disc, pmf_dice, color='#3498db', edgecolor='black', linewidth=1.2, width=0.7)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('P(X = x)', fontsize=12)
ax.set_title('Discrete: Sum of Two Dice', fontsize=13, fontweight='bold')
ax.set_xticks(x_disc)

# Continuous: normal distribution
ax = axes[1]
x_cont = np.linspace(-4, 4, 300)
pdf_normal = (1 / np.sqrt(2 * np.pi)) * np.exp(-x_cont**2 / 2)
ax.plot(x_cont, pdf_normal, 'r-', linewidth=2.5, label='PDF')
ax.fill_between(x_cont, pdf_normal, alpha=0.2, color='red')
# Shade a region for P(1 < X < 2)
mask = (x_cont >= 1) & (x_cont <= 2)
ax.fill_between(x_cont[mask], pdf_normal[mask], alpha=0.6, color='red', label='P(1 < X < 2)')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.set_title('Continuous: Standard Normal', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Discrete: P(X = 7) = 6/36 ≈", f"{6/36:.4f}")
print("Continuous: P(X = exactly 1.5) = 0  (always for continuous RVs)")
print("Continuous: P(1 < X < 2) = area under the curve (shaded region)")

---
## 4. Python: Working with Random Variables

NumPy and SciPy make it easy to define, sample from, and analyze random variables.

In [ ]:
# Using numpy for simulation
np.random.seed(0)

# Discrete RV: roll a fair die
die_rolls = np.random.randint(1, 7, size=10000)
print("Die Roll RV (10,000 samples)")
print(f"  Mean (E[X]):   {die_rolls.mean():.4f}  (theory: 3.5)")
print(f"  Variance:      {die_rolls.var():.4f}  (theory: {35/12:.4f})")
print(f"  Std Deviation: {die_rolls.std():.4f}")
print()

# Simulating a custom discrete RV
# X = max of two dice
max_two_dice = np.maximum(
    np.random.randint(1, 7, size=10000),
    np.random.randint(1, 7, size=10000)
)
print("Max of Two Dice RV (10,000 samples)")
print(f"  Mean:     {max_two_dice.mean():.4f}")
print(f"  Variance: {max_two_dice.var():.4f}")
for v in range(1, 7):
    print(f"  P(X = {v}) ≈ {np.mean(max_two_dice == v):.4f}")

In [ ]:
# Using scipy.stats for built-in distributions
from scipy import stats

# Discrete: Binomial(n=10, p=0.3) — number of successes in 10 trials
X = stats.binom(n=10, p=0.3)

print("Binomial(n=10, p=0.3):")
print(f"  E[X] = {X.mean():.2f}")
print(f"  Var(X) = {X.var():.2f}")
print(f"  P(X = 3) = {X.pmf(3):.4f}")
print(f"  P(X ≤ 3) = {X.cdf(3):.4f}")
print(f"  Sample: {X.rvs(size=10, random_state=42)}")
print()

# Continuous: Normal(μ=0, σ=1)
Y = stats.norm(loc=0, scale=1)
print("Normal(μ=0, σ=1):")
print(f"  E[Y] = {Y.mean():.2f}")
print(f"  Var(Y) = {Y.var():.2f}")
print(f"  P(Y ≤ 1.96) = {Y.cdf(1.96):.4f}")
print(f"  P(-1 < Y < 1) = {Y.cdf(1) - Y.cdf(-1):.4f}")
print(f"  Sample: {Y.rvs(size=5, random_state=42).round(3)}")

---
## Summary

| Concept | Key Point |
|---------|-----------|
| Random variable | A numerical outcome of a random experiment |
| Notation | Capital $X, Y, Z$ for RVs; lowercase $x, y, z$ for values |
| Events from RVs | Use relational operators: $P(X = 3)$, $P(X > 5)$ |
| Discrete | Countable values, uses PMF |
| Continuous | Real-valued, uses PDF |
| Key properties | Support, PMF/PDF, expectation, variance, mode |

**Next:** Probability Mass Functions — the likelihood function for discrete random variables.